# Setup and Configuration

In [ ]:
%%configure
{
"conf": {
"spark.dynamicAllocation.enabled" : "true",
"spark.dynamicAllocation.minExecutors" : 4,
"spark.dynamicAllocation.maxExecutors" : 9
}
}

In [ ]:
library(truveta.notebook.study)
library(arrow)
library(tidyverse)
library(mice)
library(SparkR)
library(lubridate)
library(Matrix)

In [ ]:
%%sparkr
con <- create_connection()
study <- get_study(con)

# Data Loading and Preprocessing

In [ ]:
all_covariates_T2D_Secondary <- load_artifacts_data(con, study, file.path("/output_data/T2D_Secondary_basedata_Mar10"))
all_covariates_T2D_Primary <- load_artifacts_data(con, study, file.path("/output_data/T2D_Primary_basedata_Mar10"))

all_covariates_T2D_Secondary <- setNames(all_covariates_T2D_Secondary, gsub("[-/\\+_ ]", "", colnames(all_covariates_T2D_Secondary)))
all_covariates_T2D_Primary <- setNames(all_covariates_T2D_Primary, gsub("[-/\\+_ ]", "", colnames(all_covariates_T2D_Primary)))

In [ ]:
print(dim(all_covariates_T2D_Secondary))
print(dim(all_covariates_T2D_Primary))

In [ ]:
head(all_covariates_T2D_Primary)

In [ ]:
colnames(all_covariates_T2D_Primary)

# Variable Classification

In [ ]:
# T2D Secondary population (use SMART score)
binary_vars <- c("HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
"HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
"HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
"HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
"HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
"HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
"HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
"HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
"HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
"HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
"HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant", "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone",
"HasOtherAntiHypertensive", "HasCerebrovascularDisease", "HasCoronaryArteryDisease", "HasPeripheralArterialDisease",
"HasAbdominalAorticAneurysm", "HasAntiHypertensiveDrugHistory", "HasStrokeHistory", "HasMiHistory"
)

categorical_vars <- c("HasHospitalization", "Sex", "Ethnicity", "Race", "CensusRegion",
 "CensusDivision", "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
 "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
 "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
 "AgeGroup", "FirstMed", "FirstMedGroup"
)

continuous_vars <- c("NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
"NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Age", "CalendarYear",
"AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue", "HsCRPValue", "HemoglobinValue",
"LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
"SystolicBloodPressureValue", "TotalCholesterolValue", "WhiteBloodCellCountValue",
"AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
"AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
"CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
"HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)

continuous_vars2 <- c("YearsSinceVascEvent")

continuous_vars3 <- c("HbA1cValue")

non_continuous <- c(binary_vars, categorical_vars)

id_vars <- c("PersonId", "FirstMedTime", "BirthDateTime")

In [ ]:
# T2D Primary population (use PREVENT score)
binary_vars <- c("HasCongestiveHeartFailure","HasCardiacArrhythmias","HasValvularHeartDisease",
"HasPulmonaryCirculationDisorders","HasPeripheralVascularDisorders","HasChronicHypertensionWithoutComplications",
"HasChronicHypertensionWithComplications","HasParalysis","HasOtherNeurologicalDisorders","HasChronicPulmonaryDisease",
"HasDiabetesUncomplicated","HasDiabetesComplicated","HasHyperthyroidism","HasRenalFailure","HasLiverDisease",
"HasPepticUlcerDisease","HasHivAids","HasLymphoma","HasMetastaticCancer","HasSolidTumorWithoutMetastasis",
"HasRheumatoidArthritis","HasCoagulopathy","HasObesityByConditionOrBmi30","HasWeightLossAndMalnutrition",
"HasFluidAndElectrolyteDisorders","HasBloodLossAnemia","HasDeficiencyAnemia","HasAlcoholAbuse","HasDrugAbuse",
"HasPsychoses","HasDepressionDiagnosis","HasSmoking", "HasDementia", "HasHyperlipidemia", "HasParaplegiaAndHemiplegia",
"HasHemorrhagicOrIschemicStroke", "HasOsteoarthritis", "HasObstructiveSleepApnea", "HasNafld",
"HasPolycysticOvarianSyndrome", "HasSchizophrenia", "HasTraumaticBrainInjury", "HasChronicObstructivePulmonaryDiseaseOrAsthma",
"HasStatin", "HasOtherLLT", "HasAntiplatelet", "HasAnticoagulant", "HasInsulin", "HasMetformin", "HasAceorArb", "HasFinerenone",
"HasOtherAntiHypertensive", "HasCerebrovascularDisease", "HasCoronaryArteryDisease", "HasPeripheralArterialDisease",
"HasAbdominalAorticAneurysm", "HasAntiHypertensiveDrugHistory", "HasStrokeHistory", "HasMiHistory"
)

categorical_vars <- c("HasHospitalization", "Sex", "Ethnicity", "Race", "CensusRegion",
 "CensusDivision", "CurrentAddressOwnOrRent","DistanceToCloseTies","CollegePastPresent",
 "HighestEduIndividual","CurrAddrDwellType","ShortLenofResBcEviction","HighestEduHousehold","HighestEduRelatives",
 "HealthcareCostRiskCategory","MedicationNonAdherenceRiskCategory","HealthManagementNonMotivationRiskCategory",
 "AgeGroup", "FirstMed", "FirstMedGroup"
)

continuous_vars <- c("NumEncounters", "NumHospitalization", "NumOutpatientVisits", "NumEmergencyVisits",
"NumAntidiabeticMed", "DcsiScore", "ElixhauserComorbidityScore", "Age", "CalendarYear",
"AltValue", "AstValue", "BmiValue", "DiastolicBloodPressureValue","EGfrValue",
"LdlCholesterolValue", "HdlCholesterolValue", "SerumCreatinineValue", "SerumTriglyceridesValue",
"SystolicBloodPressureValue", "TotalCholesterolValue",
"AddressChangeCountLast6Months", "AddressChangeCountLast12Months","HouseholdMotorizedPropertyRegistrationsCount",
"AddressChangeCountLast60Months", "FirstDegreeRelativesCount","IndividualMotorizedPropertyRegistrationsCount",
"CurrAddrMedianIncome","RaAMmbrCnt", "CurrAddrLenOfRes","HealthcareCostPercentileRank","MedicationAdherenceRate",
"HealthManagementMotivationLevel", "AddressChangeCountLast3Months", "AddressChangeCountLast1Month"
)

continuous_vars2 <- c("HbA1cValue", "HemoglobinValue", "WhiteBloodCellCountValue")

non_continuous <- c(binary_vars, categorical_vars)

id_vars <- c("PersonId", "FirstMedTime", "BirthDateTime")

In [ ]:
covariate_names <- c(binary_vars, categorical_vars, continuous_vars, continuous_vars2, continuous_vars3)
print(length(covariate_names))
print(length(unique(covariate_names)))

In [ ]:
covariate_names <- c(binary_vars, categorical_vars, continuous_vars, continuous_vars2)
print(length(covariate_names))
print(length(unique(covariate_names)))

In [ ]:
df_T2D_Secondary <- as.data.frame(all_covariates_T2D_Secondary)

id_Secondary <- which(names(df_T2D_Secondary) %in% non_continuous)

df_T2D_Secondary[, id_Secondary] <- lapply(df_T2D_Secondary[, id_Secondary, drop = FALSE], function(x) {
  factor(x, levels = unique(x))
})

In [ ]:
all_covariates_T2D_Secondary <- df_T2D_Secondary[, which(names(df_T2D_Secondary) %in%
  c(id_vars, binary_vars, categorical_vars, continuous_vars, continuous_vars2, continuous_vars3))]
print(ncol(all_covariates_T2D_Secondary))

# MICE Imputation

## Imputation Method Assignment

In [ ]:
method_T2D_Secondary <- rep("", ncol(all_covariates_T2D_Secondary))
names(method_T2D_Secondary) <- names(all_covariates_T2D_Secondary)

method_T2D_Secondary[binary_vars] <- "logreg"
method_T2D_Secondary[categorical_vars] <- "polyreg"
method_T2D_Secondary[continuous_vars] <- "pmm"
method_T2D_Secondary[continuous_vars2] <- "cart"
method_T2D_Secondary[continuous_vars3] <- "norm"

In [ ]:
method_T2D_Primary <- rep("", ncol(all_covariates_T2D_Primary))
names(method_T2D_Primary) <- names(all_covariates_T2D_Primary)

method_T2D_Primary[binary_vars] <- "logreg"
method_T2D_Primary[categorical_vars] <- "polyreg"
method_T2D_Primary[continuous_vars] <- "pmm"
method_T2D_Primary[continuous_vars2] <- "norm"

## Predictor Matrix Construction

In [ ]:
pred_matrix <- quickpred(all_covariates_T2D_Primary, mincor = 0.2)
na_rate <- colMeans(is.na(all_covariates_T2D_Secondary))
pred_matrix[, na_rate > 0.7] <- 0
pred_matrix[, id_vars] <- 0
pred_matrix[id_vars, ] <- 0

In [ ]:
pred_matrix <- quickpred(all_covariates_T2D_Primary, mincor = 0.3)
na_rate <- colMeans(is.na(all_covariates_T2D_Primary))
pred_matrix[, na_rate > 0.7] <- 0
pred_matrix[, id_vars] <- 0
pred_matrix[id_vars, ] <- 0

## Running MICE

In [ ]:
imp_T2D_Secondary <- mice(
    all_covariates_T2D_Secondary,
    method = method_T2D_Secondary,
    predictorMatrix = pred_matrix,
    m = 5,
    maxit = 3,
    ridge = 1e-3,
    printFlag = TRUE,
    seed = 123
    )

In [ ]:
imp_T2D_Primary <- mice(
    all_covariates_T2D_Primary,
    method = method_T2D_Primary,
    predictorMatrix = pred_matrix,
    m = 5,
    maxit = 3,
    ridge = 1e-3,
    printFlag = TRUE,
    seed = 123
    )

In [ ]:
completed_data <- complete(imp_T2D_Secondary, action = 1)
head(completed_data[, c("PersonId", "FirstMedTime")])

In [ ]:
sum(is.na(complete(imp_T2D_Primary)))

In [ ]:
artifacts_path <- get_artifacts_path(con, study, fs=TRUE)
dir_path <- file.path(artifacts_path, "mice")
message(dir_path)

if(! dir.exists(dir_path)) {
    dir.create(dir_path, recursive=TRUE)
    message("Directory created: ", dir_path)
}

In [ ]:
file_path_primary <- file.path(dir_path, "T2D_primary_imputed_data_Apr17.mids")
message(file_path_primary)

In [ ]:
saveRDS(imp_T2D_Primary, file = file_path_primary)

In [ ]:
artifacts_path <- get_artifacts_path(con, study, fs=TRUE)
dir_path <- file.path(artifacts_path, "mice")
files <- list.files(dir_path, full.names=TRUE)
print(files)

In [ ]:
loaded_imp_t2d_primary <- readRDS(files[1])

In [ ]:
print(dim(complete(loaded_imp_t2d_primary)))

In [ ]:
completed_list <- lapply(1:5, function(i) complete(loaded_imp_t2d_primary, i))
sapply(completed_list, function(df) sum(is.na(df)))

# Save MICE imputed datasets

In [ ]:
imputed_list <- complete(loaded_imp_t2d_primary, "all", include = FALSE)

for (i in seq_along(imputed_list)) {
  assign(paste0("final_data_", i), imputed_list[[i]])
}

In [ ]:
save_path1 <- "/mice/T2D_primary_mice_imputed_v1_for_test"
save_artifacts_data(con, study, df=final_data_1, path=save_path1, data_type="parquet", mode="overwrite")

# save_path2 <- "/mice/T2D_primary_mice_imputed_v2_for_test"
# save_artifacts_data(con, study, df=final_data_2, path=save_path2, data_type="parquet", mode="overwrite")

# save_path3 <- "/mice/T2D_primary_mice_imputed_v3_for_test"
# save_artifacts_data(con, study, df=final_data_3, path=save_path3, data_type="parquet", mode="overwrite")

# save_path4 <- "/mice/T2D_primary_mice_imputed_v4_for_test"
# save_artifacts_data(con, study, df=final_data_4, path=save_path4, data_type="parquet", mode="overwrite")

# save_path5 <- "/mice/T2D_primary_mice_imputed_v5_for_test"
# save_artifacts_data(con, study, df=final_data_5, path=save_path5, data_type="parquet", mode="overwrite")

# Post-Imputation Validation

In [ ]:
%%pyspark
t2d_imputed_df = study.load_artifacts_data("/mice/T2D_primary_mice_imputed_v5_for_test")
t2d_prior_df = study.load_artifacts_data("/output_data/T2D_Primary_basedata_Mar10")

## Missing data summary and distribution comparison

### T2D Primary Cohort

In [ ]:
%%pyspark
# T2D Primary
df = t2d_prior_df

def missing_percentage(df, show_all=False):
    missing = (df.isna().sum() / len(df) * 100).round(2)
    missing_df = missing.reset_index()
    missing_df.columns = ['Column', 'Missing (%)']

    if not show_all:
        missing_df = missing_df[missing_df['Missing (%)'] > 0]

    return missing_df.sort_values('Missing (%)', ascending=False)

missing_summary = missing_percentage(df)
missing_summary

In [ ]:
%%pyspark
import matplotlib.pyplot as plt
import seaborn as sns

variables = [
    "HbA1cValue",
    "EGfrValue",
    "LdlCholesterolValue",
    "TotalCholesterolValue",
    "HdlCholesterolValue",
    "SerumTriglyceridesValue",
    "WhiteBloodCellCountValue",
    "HemoglobinValue",
    "AstValue",
    "AltValue",
    "BmiValue",
    "SerumCreatinineValue"
]

for var in variables:
    plt.figure(figsize=(8,4))
    # Original
    sns.kdeplot(t2d_prior_df[var].dropna().to_pandas(), label="Original", fill=True)
    # Imputed
    sns.kdeplot(t2d_imputed_df[var].to_pandas(), label="Imputed", fill=True)

    plt.title(f'Distribution comparison: {var}')
    plt.xlabel(var)
    plt.ylabel("Density")
    plt.legend()
    plt.show()

### T2D Secondary Cohort

In [ ]:
%%pyspark
# T2D Secondary
df = t2d_prior_df

def missing_percentage(df, show_all=False):
    missing = (df.isna().sum() / len(df) * 100).round(2)
    missing_df = missing.reset_index()
    missing_df.columns = ['Column', 'Missing (%)']

    if not show_all:
        missing_df = missing_df[missing_df['Missing (%)'] > 0]

    return missing_df.sort_values('Missing (%)', ascending=False)

missing_summary = missing_percentage(df)
missing_summary

In [ ]:
%%pyspark
import matplotlib.pyplot as plt
import seaborn as sns

variables = [
    "LdlCholesterolValue",
    "HbA1cValue",
    "EGfrValue",
    "YearsSinceVascEvent",
    "TotalCholesterolValue",
    "HdlCholesterolValue",
    "SerumTriglyceridesValue",
    "CurrAddrMedianIncome",
    "EGfrValue",
    "IndividualMotorizedPropertyRegistrationsCount",
    "AddressChangeCountLast1Month",
    "HouseholdMotorizedPropertyRegistrationsCount"
]

for var in variables:
    plt.figure(figsize=(8,4))
    # Original
    sns.kdeplot(t2d_prior_df[var].dropna().to_pandas(), label="Original", fill=True)
    # Imputed
    sns.kdeplot(t2d_imputed_df[var].to_pandas(), label="Imputed", fill=True)

    plt.title(f'Distribution comparison: {var}')
    plt.xlabel(var)
    plt.ylabel("Density")
    plt.legend()
    plt.show()